In [10]:
from dataclasses import dataclass
from typing import List, Dict, Literal, Tuple
import pandas as pd

ContributionTiming = Literal["begin", "end"]


@dataclass
class CompoundInterestInput:
    initial_capital: float                    # Capitale iniziale
    annual_rate: float                        # Tasso annuo nominale, es. 0.07 = 7%
    years: float                              # Durata investimento in anni
    compounds_per_year: int = 12             # Frequenza capitalizzazione: 1, 4, 12, 365...
    periodic_contribution: float = 0.0       # Versamento aggiuntivo per periodo
    contribution_timing: ContributionTiming = "end"   # "begin" oppure "end"
    tax_rate: float = 0.0                    # Tassazione sui guadagni, es. 0.26 = 26%
    annual_inflation_rate: float = 0.0       # Inflazione annua, es. 0.02 = 2%


def validate_inputs(data: CompoundInterestInput) -> None:
    if data.initial_capital < 0:
        raise ValueError("Il capitale iniziale non può essere negativo.")
    if data.annual_rate < -1:
        raise ValueError("Il tasso annuo non può essere inferiore a -100%.")
    if data.years <= 0:
        raise ValueError("La durata in anni deve essere maggiore di zero.")
    if data.compounds_per_year <= 0:
        raise ValueError("La frequenza di capitalizzazione deve essere maggiore di zero.")
    if data.periodic_contribution < 0:
        raise ValueError("Il versamento periodico non può essere negativo.")
    if not (0 <= data.tax_rate <= 1):
        raise ValueError("La tassazione deve essere compresa tra 0 e 1.")
    if data.annual_inflation_rate < -1:
        raise ValueError("L'inflazione annua non può essere inferiore a -100%.")
    if data.contribution_timing not in ("begin", "end"):
        raise ValueError("contribution_timing deve essere 'begin' oppure 'end'.")


def compound_interest_calculator(data: CompoundInterestInput) -> Tuple[Dict, pd.DataFrame]:
    validate_inputs(data)

    total_periods = int(round(data.years * data.compounds_per_year))
    periodic_rate = data.annual_rate / data.compounds_per_year
    periodic_inflation_rate = data.annual_inflation_rate / data.compounds_per_year

    balance = data.initial_capital
    total_contributions = data.initial_capital
    history: List[Dict] = []

    for period in range(1, total_periods + 1):
        starting_balance = balance

        contribution_this_period = data.periodic_contribution

        if data.contribution_timing == "begin":
            balance += contribution_this_period
            total_contributions += contribution_this_period

        interest_earned = balance * periodic_rate
        balance += interest_earned

        if data.contribution_timing == "end":
            balance += contribution_this_period
            total_contributions += contribution_this_period

        history.append({
            "Periodo": period,
            "Saldo iniziale": round(starting_balance, 2),
            "Versamento": round(contribution_this_period, 2),
            "Interesse": round(interest_earned, 2),
            "Saldo finale": round(balance, 2),
        })

    gross_final_value = balance
    gross_gain = gross_final_value - total_contributions

    tax_due = max(gross_gain, 0) * data.tax_rate
    net_final_value = gross_final_value - tax_due
    net_gain = net_final_value - total_contributions

    inflation_factor = (1 + periodic_inflation_rate) ** total_periods
    real_final_value = net_final_value / inflation_factor if inflation_factor > 0 else net_final_value
    real_gain = real_final_value - total_contributions

    effective_annual_rate = (1 + periodic_rate) ** data.compounds_per_year - 1

    summary = {
        "Capitale iniziale": round(data.initial_capital, 2),
        "Totale versato": round(total_contributions, 2),
        "Valore finale lordo": round(gross_final_value, 2),
        "Guadagno lordo": round(gross_gain, 2),
        "Tasse sui guadagni": round(tax_due, 2),
        "Valore finale netto": round(net_final_value, 2),
        "Guadagno netto": round(net_gain, 2),
        "Valore reale netto, corretto per inflazione": round(real_final_value, 2),
        "Guadagno reale netto, corretto per inflazione": round(real_gain, 2),
        "Tasso effettivo annuo": round(effective_annual_rate * 100, 2),
        "Numero totale periodi": total_periods,
        "Tasso periodico": round(periodic_rate, 6),
    }

    history_df = pd.DataFrame(history)

    return summary, history_df

In [9]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import matplotlib.pyplot as plt

initial_capital = widgets.FloatText(value=10000.0, description="Capitale:")
annual_rate = widgets.FloatText(value=7.0, description="Tasso %:")
years = widgets.IntText(value=20, description="Anni:")
compounds = widgets.Dropdown(
    options=[("Annuale", 1), ("Trimestrale", 4), ("Mensile", 12)],
    value=12,
    description="Capitalizz.:"
)
periodic_contribution = widgets.FloatText(value=300.0, description="Versamento:")
contribution_timing = widgets.Dropdown(
    options=[("Fine periodo", "end"), ("Inizio periodo", "begin")],
    value="end",
    description="Versamento:"
)
tax_rate = widgets.FloatText(value=26.0, description="Tasse %:")
inflation_rate = widgets.FloatText(value=2.0, description="Inflazione %:")

button = widgets.Button(description="Calcola", button_style="success")
output = widgets.Output()

def on_calculate_clicked(b):
    with output:
        clear_output()

        params = CompoundInterestInput(
            initial_capital=initial_capital.value,
            annual_rate=annual_rate.value / 100,
            years=years.value,
            compounds_per_year=compounds.value,
            periodic_contribution=periodic_contribution.value,
            contribution_timing=contribution_timing.value,
            tax_rate=tax_rate.value / 100,
            annual_inflation_rate=inflation_rate.value / 100
        )

        summary, history_df = compound_interest_calculator(params)

        print("RISULTATI")
        for k, v in summary.items():
            print(f"{k}: € {v:,.2f}")

        display(history_df.head(10))

        plt.figure(figsize=(10, 5))
        plt.plot(history_df["Periodo"], history_df["Saldo finale"])
        plt.xlabel("Periodo")
        plt.ylabel("Saldo finale (€)")
        plt.title("Crescita dell'investimento")
        plt.grid(True)
        plt.show()

button.on_click(on_calculate_clicked)

ui = widgets.VBox([
    initial_capital,
    annual_rate,
    years,
    compounds,
    periodic_contribution,
    contribution_timing,
    tax_rate,
    inflation_rate,
    button,
    output
])

display(ui)